# Name That LLM official baseline reproduction

Word TF-IDF (1–2, 25k) + character TF-IDF (3–5, 35k), sublinear TF,
LogisticRegression(C=2.0, max_iter=1000), 5-fold GroupKFold by prompt_id.
Sources: [official Apache 2.0 Starter](https://www.kaggle.com/code/martynaplomecka/starter-notebook-for-name-that-llm-competition).
Attribution: Martyna Plomecka and Yao Yan; see repository NOTICE.md.

Attach Name That LLM inputs, select CPU and disable Internet before Save & Run All.
Uses installed pandas/numpy/scipy/scikit-learn/pyarrow/joblib/threadpoolctl, with
versions recorded in run artifacts. The exact local environment is in uv.lock.
Kaggle image parity and hidden scoring are separate checks; no score is promised.
The small visible test set is read dynamically. All original inputs stay read-only.
The package below is embedded from source, without a remote install.
Restart the kernel and use a fresh working directory for each run.


In [ ]:
# Reviewed local package sources, embedded for offline portability.
import hashlib
import json
import os
import sys
import tempfile
from pathlib import Path

if 'ntl_baseline' in sys.modules:
    raise RuntimeError('Restart the kernel before running this self-contained Notebook.')
_bundle_sources = {}
_bundle_sources['__init__.py'] = '"""Official-baseline reproduction; no data access at import time."""\n\n__version__ = "0.1.0"\n'
_bundle_sources['__main__.py'] = '"""Support python -m ntl_baseline."""\n\nfrom .cli import main\n\nraise SystemExit(main())\n'
_bundle_sources['artifacts.py'] = '"""Append-only artifacts and explicit environment/source provenance."""\n\nimport importlib.metadata\nimport json\nimport os\nimport platform\nimport resource\nimport sys\nfrom pathlib import Path\nfrom typing import Any\n\nimport pandas as pd\n\nfrom .data import file_sha256\n\nRUNTIME_PACKAGES = (\n    "numpy",\n    "scipy",\n    "pandas",\n    "scikit-learn",\n    "pyarrow",\n    "joblib",\n    "threadpoolctl",\n)\n\n\ndef write_json_new(path: Path, value: object) -> None:\n    payload = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\\n"\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("x", encoding="utf-8") as stream:\n        stream.write(payload)\n\n\ndef write_parquet_new(path: Path, frame: pd.DataFrame) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("xb") as stream:\n        frame.to_parquet(stream, engine="pyarrow", index=False)\n\n\ndef runtime_info() -> dict[str, Any]:\n    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n    return {\n        "python": sys.version.split()[0],\n        "platform": platform.platform(),\n        "machine": platform.machine(),\n        "logical_cpu_count": os.cpu_count(),\n        "packages": {name: importlib.metadata.version(name) for name in RUNTIME_PACKAGES},\n        "process_peak_rss_bytes": int(peak if sys.platform == "darwin" else peak * 1024),\n        "peak_rss_scope": "process high-water mark, not an isolated stage or process tree",\n    }\n\n\ndef source_hashes() -> dict[str, str]:\n    return {path.name: file_sha256(path) for path in sorted(Path(__file__).parent.glob("*.py"))}\n\n\ndef directory_hashes(directory: Path) -> dict[str, str]:\n    return {\n        str(path.relative_to(directory)): file_sha256(path)\n        for path in sorted(directory.rglob("*"))\n        if path.is_file()\n    }\n'
_bundle_sources['baseline.py'] = '"""Official word/character TF-IDF + logistic regression, fitted inside each fold."""\n\nfrom time import perf_counter\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom numpy.typing import NDArray\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.pipeline import FeatureUnion, Pipeline\nfrom threadpoolctl import threadpool_limits\n\nfrom .config import BaselineConfig\nfrom .constants import LABELS, ContractError\nfrom .data import validate_train\n\n\ndef build_pipeline(config: BaselineConfig) -> Pipeline:\n    return Pipeline(\n        [\n            (\n                "features",\n                FeatureUnion(\n                    [\n                        (\n                            "word_tfidf",\n                            TfidfVectorizer(\n                                ngram_range=(1, 2),\n                                max_features=config.word_max_features,\n                                sublinear_tf=True,\n                            ),\n                        ),\n                        (\n                            "char_tfidf",\n                            TfidfVectorizer(\n                                analyzer="char",\n                                ngram_range=(3, 5),\n                                max_features=config.char_max_features,\n                                sublinear_tf=True,\n                            ),\n                        ),\n                    ]\n                ),\n            ),\n            ("clf", LogisticRegression(C=config.c, max_iter=config.max_iter, solver="lbfgs")),\n        ]\n    )\n\n\ndef make_folds(train: pd.DataFrame, config: BaselineConfig) -> NDArray[np.int64]:\n    train = validate_train(train)\n    if train["prompt_id"].nunique() < config.folds:\n        raise ContractError("not enough prompt groups for requested folds")\n    folds = np.full(len(train), -1, dtype=np.int64)\n    splitter = GroupKFold(n_splits=config.folds, shuffle=False)\n    for fold, (fitting, heldout) in enumerate(\n        splitter.split(train["text"], train["model"], train["prompt_id"])\n    ):\n        fit_frame, held_frame = train.iloc[fitting], train.iloc[heldout]\n        if not set(fit_frame["prompt_id"]).isdisjoint(held_frame["prompt_id"]):\n            raise ContractError("prompt group leakage")\n        if set(fit_frame["model"]) != set(LABELS) or set(held_frame["model"]) != set(LABELS):\n            raise ContractError("every fitting and validation fold must cover all 18 labels")\n        folds[heldout] = fold\n    if (folds < 0).any():\n        raise ContractError("fold coverage incomplete")\n    return folds\n\n\ndef cross_validate(\n    train: pd.DataFrame, config: BaselineConfig\n) -> tuple[pd.DataFrame, dict[str, Any]]:\n    train = validate_train(train)\n    folds = make_folds(train, config)\n    probabilities = np.zeros((len(train), len(LABELS)), dtype=np.float64)\n    predictions = np.empty(len(train), dtype=object)\n    reports: list[dict[str, Any]] = []\n    start = perf_counter()\n    with threadpool_limits(limits=config.threads):\n        for fold in range(config.folds):\n            fitting = train.iloc[np.flatnonzero(folds != fold)]\n            heldout_indices = np.flatnonzero(folds == fold)\n            heldout = train.iloc[heldout_indices]\n            model = build_pipeline(config)\n            fold_start = perf_counter()\n            model.fit(fitting["text"], fitting["model"])\n            if tuple(model.classes_) != LABELS:\n                raise ContractError("model probability label order differs from candidate order")\n            predictions[heldout_indices] = model.predict(heldout["text"])\n            probabilities[heldout_indices] = model.predict_proba(heldout["text"])\n            reports.append(\n                {\n                    "fold": fold,\n                    "train_rows": len(fitting),\n                    "validation_rows": len(heldout),\n                    "train_prompts": int(fitting["prompt_id"].nunique()),\n                    "validation_prompts": int(heldout["prompt_id"].nunique()),\n                    "balanced_accuracy": float(\n                        balanced_accuracy_score(heldout["model"], predictions[heldout_indices])\n                    ),\n                    "seconds": perf_counter() - fold_start,\n                    "solver_iterations": int(model.named_steps["clf"].n_iter_.max()),\n                }\n            )\n    scores = np.array([report["balanced_accuracy"] for report in reports], dtype=float)\n    recalls = recall_score(train["model"], predictions, labels=LABELS, average=None)\n    oof = train[["id", "prompt_id", "model"]].copy()\n    oof["fold"] = folds\n    oof["prediction"] = predictions\n    for index, label in enumerate(LABELS):\n        oof[f"prob__{label}"] = probabilities[:, index]\n    summary = {\n        "metric": "balanced_accuracy",\n        "splitter": "GroupKFold(shuffle=False)",\n        "fold_mean_balanced_accuracy": float(scores.mean()),\n        "fold_std_balanced_accuracy": float(scores.std(ddof=0)),\n        "oof_balanced_accuracy": float(balanced_accuracy_score(train["model"], predictions)),\n        "per_class": {\n            label: {"recall": float(recalls[i]), "support": int((train["model"] == label).sum())}\n            for i, label in enumerate(LABELS)\n        },\n        "confusion_matrix_labels": list(LABELS),\n        "confusion_matrix": confusion_matrix(train["model"], predictions, labels=LABELS).tolist(),\n        "folds": reports,\n        "seconds": perf_counter() - start,\n    }\n    return oof, summary\n'
_bundle_sources['cli.py'] = '"""Explicit local CLI; no authentication, downloads or remote submissions."""\n\nimport argparse\nimport json\nimport logging\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom .artifacts import write_json_new\nfrom .config import load_config\nfrom .constants import ContractError\nfrom .data import import_archive, load_test, validate_submission\nfrom .workflow import audit_data, predict_model, run_baseline, train_model, validate_model\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    subparsers = parser.add_subparsers(dest="command", required=True)\n    for command in ("audit", "validate", "train", "predict", "run", "check-submission"):\n        subparser = subparsers.add_parser(command)\n        subparser.add_argument("--data-dir", type=Path, required=True)\n        if command in ("validate", "train", "run"):\n            subparser.add_argument("--config", type=Path)\n        if command in ("validate", "train", "predict"):\n            subparser.add_argument("--output-dir", type=Path, required=True)\n        if command == "predict":\n            subparser.add_argument(\n                "--model-dir",\n                type=Path,\n                required=True,\n                help="trusted locally trained joblib artifacts only",\n            )\n        if command == "run":\n            subparser.add_argument("--run-dir", type=Path, required=True)\n        if command == "audit":\n            subparser.add_argument("--output", type=Path)\n        if command == "check-submission":\n            subparser.add_argument("--submission", type=Path, required=True)\n    importer = subparsers.add_parser("import-data")\n    importer.add_argument("--archive", type=Path, required=True)\n    importer.add_argument("--data-dir", type=Path, default=Path("data/raw/name-that-llm"))\n    args = parser.parse_args(argv)\n    logging.basicConfig(level=logging.INFO, stream=sys.stderr, format="%(message)s")\n    try:\n        if args.command == "import-data":\n            result = import_archive(args.archive, args.data_dir)\n        elif args.command == "audit":\n            result = audit_data(args.data_dir)\n            if args.output:\n                write_json_new(args.output, result)\n        elif args.command == "validate":\n            result = validate_model(args.data_dir, args.output_dir, load_config(args.config))\n        elif args.command == "train":\n            result = train_model(args.data_dir, args.output_dir, load_config(args.config))\n        elif args.command == "predict":\n            result = predict_model(args.data_dir, args.model_dir, args.output_dir)\n        elif args.command == "run":\n            result = run_baseline(args.data_dir, args.run_dir, load_config(args.config))\n        else:\n            test = load_test(args.data_dir)\n            submission = validate_submission(pd.read_parquet(args.submission), test)\n            result = {"status": "valid", "rows": len(submission)}\n    except (ContractError, FileNotFoundError, FileExistsError) as exc:\n        print(f"error: {exc}", file=sys.stderr)\n        return 2\n    print(json.dumps(result, indent=2, ensure_ascii=False, allow_nan=False))\n    return 0\n'
_bundle_sources['config.py'] = '"""Configuration for the official baseline (not tuned)."""\n\nimport json\nimport math\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nfrom .constants import ContractError\n\n\n@dataclass(frozen=True)\nclass BaselineConfig:\n    word_max_features: int = 25_000\n    char_max_features: int = 35_000\n    c: float = 2.0\n    max_iter: int = 1_000\n    folds: int = 5\n    threads: int = 1\n\n    def __post_init__(self) -> None:\n        for name in ("word_max_features", "char_max_features", "max_iter", "threads", "folds"):\n            value = getattr(self, name)\n            if isinstance(value, bool) or not isinstance(value, int) or value < 1:\n                raise ContractError(f"config {name} must be a positive integer")\n        if self.folds < 2:\n            raise ContractError("config folds must be at least 2")\n        if isinstance(self.c, bool) or not isinstance(self.c, int | float):\n            raise ContractError("config c must be finite and positive")\n        if not math.isfinite(self.c) or self.c <= 0:\n            raise ContractError("config c must be finite and positive")\n\n\ndef load_config(path: Path | None) -> BaselineConfig:\n    if path is None:\n        return BaselineConfig()\n    try:\n        values = json.loads(path.read_text(encoding="utf-8"))\n        if not isinstance(values, dict):\n            raise ContractError("config must be a JSON object")\n        return BaselineConfig(**values)\n    except (json.JSONDecodeError, TypeError) as exc:\n        raise ContractError("invalid config keys or JSON") from exc\n'
_bundle_sources['constants.py'] = '"""Exact prediction labels from the competition\'s Evaluation page."""\n\nLABELS: tuple[str, ...] = tuple(\n    sorted(\n        [\n            "gpt-5.5",\n            "gpt-5.6-luna",\n            "gpt-5.6-sol",\n            "gpt-5.6-terra",\n            "gpt-6-astra",\n            "gemini-3.1-pro-preview",\n            "gemini-3.6-flash",\n            "gemini-3.7-flash",\n            "gemini-3.8-flash",\n            "claude-fable-5",\n            "claude-opus-4-6",\n            "claude-opus-4-8",\n            "claude-opus-5",\n            "claude-sonnet-5",\n            "qwen3.8-max",\n            "glm-5.3",\n            "glm-5.3-flash",\n            "kimi-k3",\n        ]\n    )\n)\n\nDATA_FILES = ("train.csv", "test.csv", "sample_submission.parquet")\n\n\nclass ContractError(ValueError):\n    """Invalid data, configuration, artifact or submission contract."""\n'
_bundle_sources['data.py'] = '"""Local-only input, archive and submission contracts. Never log passage content."""\n\nimport csv\nimport hashlib\nimport json\nimport stat\nimport tempfile\nimport zipfile\nfrom datetime import UTC, datetime\nfrom pathlib import Path, PurePosixPath\n\nimport pandas as pd\n\nfrom .constants import DATA_FILES, LABELS, ContractError\n\nMAX_INPUT_BYTES = 64 * 1024 * 1024\n\n\ndef file_sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef _check_columns(frame: pd.DataFrame, expected: tuple[str, ...]) -> None:\n    if not frame.columns.is_unique or set(frame.columns) != set(expected):\n        raise ContractError(f"columns must be exactly {\', \'.join(expected)}")\n    if frame.empty:\n        raise ContractError("input must contain at least one row")\n\n\ndef _check_strings(series: pd.Series, column: str) -> None:\n    if (\n        series.isna().any()\n        or not series.map(lambda value: isinstance(value, str) and bool(value.strip())).all()\n    ):\n        raise ContractError(f"{column} must contain non-empty strings")\n\n\ndef _check_ids(frame: pd.DataFrame) -> None:\n    _check_strings(frame["id"], "id")\n    if frame["id"].duplicated().any():\n        raise ContractError("id must be unique")\n\n\ndef _normalize_text(frame: pd.DataFrame) -> pd.DataFrame:\n    result = frame.copy(deep=True).reset_index(drop=True)\n    result["text"] = result["text"].fillna("")\n    if not result["text"].map(lambda value: isinstance(value, str)).all():\n        raise ContractError("text must contain strings or missing values")\n    return result\n\n\ndef validate_train(frame: pd.DataFrame) -> pd.DataFrame:\n    _check_columns(frame, ("id", "prompt_id", "text", "model"))\n    _check_ids(frame)\n    _check_strings(frame["prompt_id"], "prompt_id")\n    _check_strings(frame["model"], "model label")\n    if not frame["model"].isin(LABELS).all():\n        raise ContractError("unknown model label")\n    if set(frame["model"]) != set(LABELS):\n        raise ContractError("training data must cover all 18 labels")\n    return _normalize_text(frame)\n\n\ndef validate_test(frame: pd.DataFrame) -> pd.DataFrame:\n    _check_columns(frame, ("id", "text"))\n    _check_ids(frame)\n    return _normalize_text(frame)\n\n\ndef validate_submission(frame: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:\n    test = validate_test(test)\n    _check_columns(frame, ("id", "model"))\n    _check_ids(frame)\n    _check_strings(frame["model"], "model label")\n    if not frame["model"].isin(LABELS).all():\n        raise ContractError("submission contains an invalid model label")\n    if len(frame) != len(test) or set(frame["id"]) != set(test["id"]):\n        raise ContractError("submission id set must equal test id set")\n    return frame.set_index("id").loc[test["id"], ["model"]].reset_index()\n\n\ndef input_file(data_dir: Path, name: str) -> Path:\n    path = data_dir / name\n    if path.is_symlink() or not path.is_file():\n        raise ContractError(f"missing regular input file: {name}")\n    if path.stat().st_size > MAX_INPUT_BYTES:\n        raise ContractError(f"input exceeds local size budget: {name}")\n    return path\n\n\ndef read_csv_input(data_dir: Path, name: str) -> pd.DataFrame:\n    path = input_file(data_dir, name)\n    try:\n        with path.open(encoding="utf-8", newline="") as stream:\n            header = next(csv.reader(stream), [])\n        if not header or len(set(header)) != len(header):\n            raise ContractError(f"invalid or duplicate columns in {name}")\n        # Enforce the published string schema (including leading-zero IDs).\n        # Missing text is still normalized with the official fillna(\'\') policy.\n        return pd.read_csv(path, dtype={name: str for name in ("id", "prompt_id", "text", "model")})\n    except (pd.errors.ParserError, pd.errors.EmptyDataError, UnicodeError) as exc:\n        raise ContractError(f"invalid CSV input: {name}") from exc\n\n\ndef load_test(data_dir: Path) -> pd.DataFrame:\n    return validate_test(read_csv_input(data_dir, "test.csv"))\n\n\ndef load_competition_data(\n    data_dir: Path, *, require_sample: bool = True\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    train = validate_train(read_csv_input(data_dir, "train.csv"))\n    test = load_test(data_dir)\n    if not require_sample:\n        # Hidden reruns replace test.csv; the public sample is not its ID authority.\n        return train, test\n    sample_path = input_file(data_dir, "sample_submission.parquet")\n    try:\n        sample = pd.read_parquet(sample_path, engine="pyarrow")\n    except (ValueError, OSError) as exc:\n        raise ContractError("invalid sample_submission.parquet") from exc\n    validate_submission(sample, test)\n    return train, test\n\n\ndef import_archive(archive: Path, destination: Path) -> dict[str, object]:\n    if destination.exists() and (not destination.is_dir() or any(destination.iterdir())):\n        raise FileExistsError(f"refusing to replace existing data: {destination}")\n    if destination.is_symlink() or not archive.is_file():\n        raise ContractError("archive and destination must be local regular paths")\n    if archive.stat().st_size > MAX_INPUT_BYTES:\n        raise ContractError("archive exceeds local size budget")\n    try:\n        with zipfile.ZipFile(archive) as bundle:\n            members = bundle.infolist()\n            names = [member.filename for member in members]\n            for member in members:\n                path = PurePosixPath(member.filename)\n                is_link = stat.S_ISLNK(member.external_attr >> 16)\n                if path.is_absolute() or ".." in path.parts or "\\\\" in member.filename or is_link:\n                    raise ContractError("unsafe archive member path or symlink")\n            if len(members) != 3 or set(names) != set(DATA_FILES):\n                raise ContractError("archive must contain exactly the three official files at root")\n            if sum(member.file_size for member in members) > MAX_INPUT_BYTES:\n                raise ContractError("archive expansion exceeds local size budget")\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            with tempfile.TemporaryDirectory(prefix=".ntl-import-", dir=destination.parent) as temp:\n                staging = Path(temp) / "data"\n                staging.mkdir()\n                for member in members:\n                    payload = bundle.read(member)\n                    if len(payload) != member.file_size:\n                        raise ContractError("archive member size mismatch")\n                    (staging / member.filename).write_bytes(payload)\n                load_competition_data(staging)\n                manifest: dict[str, object] = {\n                    "source_url": "https://www.kaggle.com/competitions/name-that-llm/data",\n                    "imported_at_utc": datetime.now(UTC).isoformat(),\n                    "archive_name": archive.name,\n                    "archive_sha256": file_sha256(archive),\n                    "files": {name: file_sha256(staging / name) for name in DATA_FILES},\n                }\n                (staging / "source.json").write_text(\n                    json.dumps(manifest, indent=2) + "\\n", encoding="utf-8"\n                )\n                staging.rename(destination)\n                return manifest\n    except zipfile.BadZipFile as exc:\n        raise ContractError("invalid ZIP archive") from exc\n'
_bundle_sources['notebook.py'] = '"""Generate a self-contained Notebook from the same reviewed package sources."""\n\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .artifacts import source_hashes\n\n\ndef _cell(kind: str, source: str, cell_id: str) -> dict[str, Any]:\n    result: dict[str, Any] = {\n        "cell_type": kind,\n        "id": cell_id,\n        "metadata": {},\n        "source": source,\n    }\n    if kind == "code":\n        result.update({"execution_count": None, "outputs": []})\n    return result\n\n\ndef notebook_document() -> dict[str, Any]:\n    sources = {\n        path.name: path.read_text(encoding="utf-8")\n        for path in sorted(Path(__file__).parent.glob("*.py"))\n    }\n    bootstrap = """# Reviewed local package sources, embedded for offline portability.\nimport hashlib\nimport json\nimport os\nimport sys\nimport tempfile\nfrom pathlib import Path\n\nif \'ntl_baseline\' in sys.modules:\n    raise RuntimeError(\'Restart the kernel before running this self-contained Notebook.\')\n_bundle_sources = {}\n"""\n    bootstrap += "\\n".join(\n        f"_bundle_sources[{name!r}] = {source!r}" for name, source in sources.items()\n    )\n    bootstrap += f"\\n_expected_source_hashes = {source_hashes()!r}\\n"\n    bootstrap += """\nfor _name, _source in _bundle_sources.items():\n    if hashlib.sha256(_source.encode(\'utf-8\')).hexdigest() != _expected_source_hashes[_name]:\n        raise RuntimeError(\'Embedded source checksum mismatch\')\n_bundle_tmp = tempfile.TemporaryDirectory(prefix=\'ntl-baseline-\')\n_package_dir = Path(_bundle_tmp.name) / \'ntl_baseline\'\n_package_dir.mkdir()\nfor _name, _source in _bundle_sources.items():\n    (_package_dir / _name).write_text(_source, encoding=\'utf-8\')\nsys.path.insert(0, _bundle_tmp.name)\n\nfrom ntl_baseline.config import BaselineConfig\nfrom ntl_baseline.data import load_test, validate_submission\nfrom ntl_baseline.workflow import run_baseline\n"""\n    configuration = """# Only explicit, known competition mounts; no recursive first-match lookup.\nif os.environ.get(\'NTL_DATA_DIR\'):\n    DATA_DIR = Path(os.environ[\'NTL_DATA_DIR\'])\nelse:\n    _candidates = [\n        Path(\'/kaggle/input/competitions/name-that-llm\'),\n        Path(\'/kaggle/input/name-that-llm\'),\n    ]\n    _existing = [p for p in _candidates if (p / \'train.csv\').is_file()]\n    if len(_existing) != 1:\n        raise RuntimeError(\'Attach the competition input or explicitly set NTL_DATA_DIR.\')\n    DATA_DIR = _existing[0]\nRUN_DIR = Path(os.environ.get(\'NTL_RUN_DIR\', str(Path.cwd() / \'baseline-run\')))\nCONFIG = BaselineConfig()\n"""\n    execute = """# Fresh directory required. No network calls, installs or pretrained weights.\n# The public sample may be absent or stale during hidden-test reruns.\n# Final predictions are still checked against every ID in the actual test.csv.\nresult = run_baseline(DATA_DIR, RUN_DIR, CONFIG, require_sample=False)\nprint(json.dumps(result, indent=2))\n"""\n    export = """import pandas as pd\n\n_submission_source = RUN_DIR / \'prediction\' / \'submission.parquet\'\nSUBMISSION_PATH = Path.cwd() / \'submission.parquet\'\nwith SUBMISSION_PATH.open(\'xb\') as _out:\n    _out.write(_submission_source.read_bytes())\n_checked = validate_submission(pd.read_parquet(SUBMISSION_PATH), load_test(DATA_DIR))\nprint(json.dumps({\'submission_file\': SUBMISSION_PATH.name, \'rows\': len(_checked)}))\n_bundle_tmp.cleanup()\n"""\n    return {\n        "nbformat": 4,\n        "nbformat_minor": 5,\n        "metadata": {\n            "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},\n            "language_info": {"name": "python"},\n            "ntl_baseline_source_sha256": source_hashes(),\n        },\n        "cells": [\n            _cell(\n                "markdown",\n                """# Name That LLM official baseline reproduction\n\nWord TF-IDF (1–2, 25k) + character TF-IDF (3–5, 35k), sublinear TF,\nLogisticRegression(C=2.0, max_iter=1000), 5-fold GroupKFold by prompt_id.\nSources: [official Apache 2.0 Starter](https://www.kaggle.com/code/martynaplomecka/starter-notebook-for-name-that-llm-competition).\nAttribution: Martyna Plomecka and Yao Yan; see repository NOTICE.md.\n\nAttach Name That LLM inputs, select CPU and disable Internet before Save & Run All.\nUses installed pandas/numpy/scipy/scikit-learn/pyarrow/joblib/threadpoolctl, with\nversions recorded in run artifacts. The exact local environment is in uv.lock.\nKaggle image parity and hidden scoring are separate checks; no score is promised.\nThe small visible test set is read dynamically. All original inputs stay read-only.\nThe package below is embedded from source, without a remote install.\nRestart the kernel and use a fresh working directory for each run.\n""",\n                "overview",\n            ),\n            _cell("code", bootstrap, "package-bootstrap"),\n            _cell("code", configuration, "configuration"),\n            _cell("code", execute, "train-validate-predict"),\n            _cell("code", export, "export-submission"),\n        ],\n    }\n'
_bundle_sources['workflow.py'] = '"""Audited validation, training and prediction on immutable local input files."""\n\nimport importlib.metadata\nimport json\nimport logging\nfrom dataclasses import asdict\nfrom datetime import UTC, datetime\nfrom pathlib import Path\nfrom time import perf_counter\nfrom typing import Any\n\nimport joblib\nimport pandas as pd\nfrom sklearn.pipeline import Pipeline\nfrom threadpoolctl import threadpool_limits\n\nfrom .artifacts import (\n    directory_hashes,\n    runtime_info,\n    source_hashes,\n    write_json_new,\n    write_parquet_new,\n)\nfrom .baseline import build_pipeline, cross_validate\nfrom .config import BaselineConfig\nfrom .constants import DATA_FILES, LABELS, ContractError\nfrom .data import (\n    file_sha256,\n    input_file,\n    load_competition_data,\n    load_test,\n    read_csv_input,\n    validate_submission,\n    validate_train,\n)\n\nLOGGER = logging.getLogger(__name__)\n\n\ndef audit_data(data_dir: Path, *, require_sample: bool = True) -> dict[str, Any]:\n    train, test = load_competition_data(data_dir, require_sample=require_sample)\n    audited_files = DATA_FILES if require_sample else ("train.csv", "test.csv")\n    files = {\n        name: {\n            "sha256": file_sha256(input_file(data_dir, name)),\n            "bytes": input_file(data_dir, name).stat().st_size,\n        }\n        for name in audited_files\n    }\n    if (data_dir / "source.json").is_file():\n        provenance = json.loads((data_dir / "source.json").read_text(encoding="utf-8"))\n        expected_hashes = provenance.get("files", {})\n        if any(expected_hashes.get(name) != files[name]["sha256"] for name in audited_files):\n            raise ContractError("raw input hash differs from import manifest")\n    words = train["text"].str.split().str.len()\n    return {\n        "files": files,\n        "train_rows": len(train),\n        "test_rows": len(test),\n        "prompt_count": int(train["prompt_id"].nunique()),\n        "candidate_count": int(train["model"].nunique()),\n        "train_class_counts": {label: int((train["model"] == label).sum()) for label in LABELS},\n        "prompt_size_counts": {\n            str(size): int(count)\n            for size, count in train.groupby("prompt_id").size().value_counts().items()\n        },\n        "train_empty_text_rows": int((train["text"] == "").sum()),\n        "test_empty_text_rows": int((test["text"] == "").sum()),\n        "train_word_count_range": [int(words.min()), int(words.max())],\n        "train_duplicate_text_rows": int(train["text"].duplicated().sum()),\n        "train_test_exact_text_overlap": int(test["text"].isin(set(train["text"])).sum()),\n        "matches_published_training_shape": len(train) == 2699\n        and train["prompt_id"].nunique() == 150,\n        "text_policy": "published string schema; official missing text filled with empty string",\n    }\n\n\ndef validate_model(data_dir: Path, output_dir: Path, config: BaselineConfig) -> dict[str, Any]:\n    train = validate_train(read_csv_input(data_dir, "train.csv"))\n    before = file_sha256(data_dir / "train.csv")\n    output_dir.mkdir(parents=True, exist_ok=False)\n    LOGGER.info("Running %d prompt-grouped validation folds on %d rows", config.folds, len(train))\n    oof, summary = cross_validate(train, config)\n    if file_sha256(data_dir / "train.csv") != before:\n        raise ContractError("training input changed during validation")\n    write_parquet_new(output_dir / "oof.parquet", oof)\n    write_parquet_new(output_dir / "folds.parquet", oof[["id", "prompt_id", "fold"]])\n    summary.update(\n        {\n            "train_sha256": before,\n            "config": asdict(config),\n            "runtime": runtime_info(),\n            "source_sha256": source_hashes(),\n        }\n    )\n    write_json_new(output_dir / "metrics.json", summary)\n    return summary\n\n\ndef train_model(data_dir: Path, model_dir: Path, config: BaselineConfig) -> dict[str, Any]:\n    train = validate_train(read_csv_input(data_dir, "train.csv"))\n    before = file_sha256(data_dir / "train.csv")\n    model_dir.mkdir(parents=True, exist_ok=False)\n    model = build_pipeline(config)\n    LOGGER.info("Fitting full-training model on %d rows", len(train))\n    start = perf_counter()\n    with threadpool_limits(limits=config.threads):\n        model.fit(train["text"], train["model"])\n    if tuple(model.classes_) != LABELS:\n        raise ContractError("fitted model must cover all 18 labels in canonical order")\n    if file_sha256(data_dir / "train.csv") != before:\n        raise ContractError("training input changed during fitting")\n    path = model_dir / "pipeline.joblib"\n    with path.open("xb") as stream:\n        joblib.dump(model, stream)\n    metadata = {\n        "format_version": 1,\n        "labels": list(LABELS),\n        "config": asdict(config),\n        "train_rows": len(train),\n        "train_sha256": before,\n        "model_sha256": file_sha256(path),\n        "seconds": perf_counter() - start,\n        "solver_iterations": int(model.named_steps["clf"].n_iter_.max()),\n        "runtime": runtime_info(),\n        "source_sha256": source_hashes(),\n        "trusted_local_artifact_only": True,\n    }\n    write_json_new(model_dir / "metadata.json", metadata)\n    return metadata\n\n\ndef load_trusted_model(model_dir: Path) -> tuple[Pipeline, dict[str, Any]]:\n    # joblib is pickle-based. Integrity checks do NOT make third-party pickle safe.\n    # Only use artifacts trained by this project from trusted local inputs.\n    metadata = json.loads((model_dir / "metadata.json").read_text(encoding="utf-8"))\n    if metadata.get("format_version") != 1 or metadata.get("labels") != list(LABELS):\n        raise ContractError("incompatible model metadata or labels")\n    for name, version in metadata["runtime"]["packages"].items():\n        if importlib.metadata.version(name) != version:\n            raise ContractError(f"model runtime package version mismatch: {name}")\n    model_path = model_dir / "pipeline.joblib"\n    if file_sha256(model_path) != metadata.get("model_sha256"):\n        raise ContractError("model hash mismatch; refusing to deserialize")\n    model = joblib.load(model_path)\n    if not isinstance(model, Pipeline) or tuple(model.classes_) != LABELS:\n        raise ContractError("invalid fitted pipeline or probability label order")\n    return model, metadata\n\n\ndef predict_model(data_dir: Path, model_dir: Path, output_dir: Path) -> dict[str, Any]:\n    test = load_test(data_dir)\n    before = file_sha256(data_dir / "test.csv")\n    model, metadata = load_trusted_model(model_dir)\n    output_dir.mkdir(parents=True, exist_ok=False)\n    start = perf_counter()\n    with threadpool_limits(limits=metadata["config"]["threads"]):\n        predictions = model.predict(test["text"])\n    submission = validate_submission(pd.DataFrame({"id": test["id"], "model": predictions}), test)\n    path = output_dir / "submission.parquet"\n    write_parquet_new(path, submission)\n    actual = pd.read_parquet(path, engine="pyarrow")\n    validate_submission(actual, test)\n    pd.testing.assert_frame_equal(actual, submission)\n    if file_sha256(data_dir / "test.csv") != before:\n        raise ContractError("test input changed during inference")\n    report = {\n        "test_rows": len(test),\n        "test_sha256": before,\n        "submission_sha256": file_sha256(path),\n        "model_sha256": metadata["model_sha256"],\n        "seconds": perf_counter() - start,\n        "prediction_counts": {label: int((actual["model"] == label).sum()) for label in LABELS},\n        "parquet_roundtrip_validated": True,\n        "runtime": runtime_info(),\n    }\n    write_json_new(output_dir / "report.json", report)\n    return report\n\n\ndef run_baseline(\n    data_dir: Path, run_dir: Path, config: BaselineConfig, *, require_sample: bool = True\n) -> dict[str, Any]:\n    initial_audit = audit_data(data_dir, require_sample=require_sample)\n    run_dir.mkdir(parents=True, exist_ok=False)\n    start = perf_counter()\n    write_json_new(\n        run_dir / "started.json",\n        {\n            "started_at_utc": datetime.now(UTC).isoformat(),\n            "config": asdict(config),\n            "source_sha256": source_hashes(),\n            "runtime": runtime_info(),\n        },\n    )\n    write_json_new(run_dir / "data_manifest.json", initial_audit)\n    validation = validate_model(data_dir, run_dir / "validation", config)\n    training = train_model(data_dir, run_dir / "model", config)\n    prediction = predict_model(data_dir, run_dir / "model", run_dir / "prediction")\n    final_audit = audit_data(data_dir, require_sample=require_sample)\n    if initial_audit != final_audit:\n        raise ContractError("raw inputs changed during run; no completion record written")\n    result = {\n        "status": "complete",\n        "completed_at_utc": datetime.now(UTC).isoformat(),\n        "train_rows": initial_audit["train_rows"],\n        "test_rows": initial_audit["test_rows"],\n        "fold_mean_balanced_accuracy": validation["fold_mean_balanced_accuracy"],\n        "fold_std_balanced_accuracy": validation["fold_std_balanced_accuracy"],\n        "oof_balanced_accuracy": validation["oof_balanced_accuracy"],\n        "validation_seconds": validation["seconds"],\n        "training_seconds": training["seconds"],\n        "prediction_seconds": prediction["seconds"],\n        "seconds": perf_counter() - start,\n        "raw_input_hashes_unchanged": True,\n        "submission": "prediction/submission.parquet",\n        "runtime": runtime_info(),\n        "artifact_sha256": directory_hashes(run_dir),\n    }\n    write_json_new(run_dir / "completed.json", result)\n    return result\n'
_expected_source_hashes = {'__init__.py': '534e449b7b5d310d159db644162ba10f7f2c0e0ec33f616e4cb954be1d072639', '__main__.py': 'c92736af4dda33ba09b0620aad3f832a605494ce24cff7a763d0742d0131698d', 'artifacts.py': '2c20c54cfdeecdca179a97e9e1462d8dd0fa42ef3436d11b3c6fd4141abd66e9', 'baseline.py': 'e3c802e1c45dc2a677fd83444ef0502111f248afededa566adf795612a6dadb8', 'cli.py': '7e8fba7c13b1880df5cbf862ca93ffdf7b1e4f5b67277650d425d110d3829d2b', 'config.py': 'eb3f12fe6867722a0df1bcac217f2b2e6749886d26cdebc5fbac749a5c81b909', 'constants.py': '9149e8db8973ffdca1cc67452ca105efc81c1ea4677ae8cbb9d7f497d6c661bc', 'data.py': '0d03ef59f14f60ce7f4a96d999292e6753bc1763455734c59fba204acbb66c7e', 'notebook.py': '3bf6a8a213209698d3d32f73ed15293ead9edf2fd549fe30ff7e33937b4c074c', 'workflow.py': 'd4a0465255c5291c9fc80170a2c155552840d3d6dd8bda099474e80c5b347db4'}

for _name, _source in _bundle_sources.items():
    if hashlib.sha256(_source.encode('utf-8')).hexdigest() != _expected_source_hashes[_name]:
        raise RuntimeError('Embedded source checksum mismatch')
_bundle_tmp = tempfile.TemporaryDirectory(prefix='ntl-baseline-')
_package_dir = Path(_bundle_tmp.name) / 'ntl_baseline'
_package_dir.mkdir()
for _name, _source in _bundle_sources.items():
    (_package_dir / _name).write_text(_source, encoding='utf-8')
sys.path.insert(0, _bundle_tmp.name)

from ntl_baseline.config import BaselineConfig
from ntl_baseline.data import load_test, validate_submission
from ntl_baseline.workflow import run_baseline


In [ ]:
# Only explicit, known competition mounts; no recursive first-match lookup.
if os.environ.get('NTL_DATA_DIR'):
    DATA_DIR = Path(os.environ['NTL_DATA_DIR'])
else:
    _candidates = [
        Path('/kaggle/input/competitions/name-that-llm'),
        Path('/kaggle/input/name-that-llm'),
    ]
    _existing = [p for p in _candidates if (p / 'train.csv').is_file()]
    if len(_existing) != 1:
        raise RuntimeError('Attach the competition input or explicitly set NTL_DATA_DIR.')
    DATA_DIR = _existing[0]
RUN_DIR = Path(os.environ.get('NTL_RUN_DIR', str(Path.cwd() / 'baseline-run')))
CONFIG = BaselineConfig()


In [ ]:
# Fresh directory required. No network calls, installs or pretrained weights.
# The public sample may be absent or stale during hidden-test reruns.
# Final predictions are still checked against every ID in the actual test.csv.
result = run_baseline(DATA_DIR, RUN_DIR, CONFIG, require_sample=False)
print(json.dumps(result, indent=2))


In [ ]:
import pandas as pd

_submission_source = RUN_DIR / 'prediction' / 'submission.parquet'
SUBMISSION_PATH = Path.cwd() / 'submission.parquet'
with SUBMISSION_PATH.open('xb') as _out:
    _out.write(_submission_source.read_bytes())
_checked = validate_submission(pd.read_parquet(SUBMISSION_PATH), load_test(DATA_DIR))
print(json.dumps({'submission_file': SUBMISSION_PATH.name, 'rows': len(_checked)}))
_bundle_tmp.cleanup()
